# CitationRAG - Week 5 API Notebook
**Applied GenAI & Agentic AI Engineering Course · Week 5**

CitationRAG answers questions from the Week 4 KnowledgeVault (Qdrant) with inline source citations,
validates every citation deterministically, and refuses gracefully when context is too weak.
This notebook covers every endpoint two ways - cURL and Python.

---
### Before you start
1. Make sure Week 4 KnowledgeVault ingestion is complete - the `qdrant_local/` folder in your week-04 project has data (this notebook targets the Week 4 lab index: the Attention paper)
2. Copy `.env.example` to `.env`, set `OPENAI_API_KEY`, and point `QDRANT_LOCAL_PATH` at that `qdrant_local/` folder (or copy the folder into this project). Stop the Week 4 server first - the embedded store allows one process at a time.
3. Generate the golden eval dataset: `python scripts/build_golden_dataset.py`
4. Start the server: `uvicorn app.main:app --reload`
5. Run the **Setup** cell below once.

> **Shell note:** the curl cells use `!curl` with `\"` to escape inner quotes - the same line works in Windows cmd, bash and zsh.

In [ ]:
# Setup -- run this cell first
import requests, json

BASE = 'http://localhost:8000'

# Primary demo question -- ASCII only, used verbatim in the !curl cells.
DEMO_NOTES = 'What is self-attention and how does it work?'

# Additional demo variants (all about the Attention Is All You Need paper)
DEMO = {
    'self_attn':  'What is self-attention and how does it work?',
    'multihead':  'What does multi-head attention allow a model to do?',
    'bleu':       'What BLEU score did the Transformer achieve on English-to-German?',
    'pos_enc':    'What is positional encodings?',
    'offTopic':   'What is the lunchroom Wi-Fi password?',
    'offTopic2':  'What is the current stock price of NVIDIA?',
}

# RECIPIENT -- not used this week (no email endpoint).
# RECIPIENT = 'you@example.com'

print('Setup complete.')
print('BASE:', BASE)

---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [ ]:
!curl -s http://localhost:8000/health

In [ ]:
# Health check -- Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 2 · Ask a Covered Question - `POST /answer`

The `/answer` endpoint runs the four-stage pipeline:
**retrieve** top-k chunks from Qdrant (dense + BM25 + RRF) -> **threshold gate** -> **generate** with citation contract -> **validate** citations.

Key concept: every factual claim in the answer is followed by `[doc#N]` where N is a chunk ID
sent to the model. The validator checks this deterministically after generation.

Request body:
```json
{ "question": "string (min 3 chars)" }
```

Response shape:
```json
{
  "answer": "...prose with [doc#N] markers...",
  "citations": [{"chunk_id": "doc#N", "supporting_quote": "..."}],
  "refused": false,
  "retrieval_top_score": 0.78,
  "retrieval_spread": 0.17,
  "validation_passed": true
}
```

> Requires `OPENAI_API_KEY`, `QDRANT_URL`, and `QDRANT_API_KEY` in `.env`.
> The knowledge index lives in the Week 4 Qdrant `knowledgevault` collection.

In [ ]:
!curl -s -X POST http://localhost:8000/answer -H "Content-Type: application/json" -d "{\"question\": \"What is self-attention and how does it work?\"}"

In [ ]:
# POST /answer -- covered question about the Attention paper
r = requests.post(f'{BASE}/answer', json={'question': DEMO['self_attn']})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('refused         :', data['refused'])
    print('validation_passed:', data['validation_passed'])
    print('retrieval_top    :', data['retrieval_top_score'])
    print('citations        :', len(data['citations']))
    print()
    print('-- Answer --')
    print(data['answer'])
    print()
    print('-- Citations --')
    for c in data['citations']:
        print(f"  [{c['chunk_id']}] {c['supporting_quote'][:80]}...")

---
## 3 · Ask an Off-Topic Question - Refusal Path

When the knowledge index does not cover the question, the retrieval scores drop below
threshold OR the LLM sees irrelevant chunks and returns the exact refusal string.
`refused=true`, empty citations list.

Key concept: the refusal string is **byte-identical** every time -
`"I don't have that information in the provided sources."` - so downstream code
can detect it deterministically without parsing prose.

In [ ]:
!curl -s -X POST http://localhost:8000/answer -H "Content-Type: application/json" -d "{\"question\": \"What is the lunchroom Wi-Fi password?\"}"

In [ ]:
# POST /answer -- off-topic question (expect refused=True)
r = requests.post(f'{BASE}/answer', json={'question': DEMO['offTopic']})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('refused         :', data['refused'])
    print('refusal_source  :', data.get('refusal_source'))
    print('citations       :', data['citations'])  # should be []
    print('answer          :', data['answer'])

---
## 4 · Two Questions Side by Side

Compare covered vs. off-topic questions to see the answered/refused split.
Also shows `refusal_source` which tells you which stage refused and why.

In [ ]:
# Side-by-side: answered vs refused
questions = [
    ('self-attn',  DEMO['self_attn']),
    ('multihead',  DEMO['multihead']),
    ('off-topic',  DEMO['offTopic']),
    ('off-topic2', DEMO['offTopic2']),
]

print(f'{"label":<12} {"refused":<9} {"source":<14} {"top_score":<11} question')
print('-' * 85)
for label, q in questions:
    r = requests.post(f'{BASE}/answer', json={'question': q})
    if r.status_code != 200:
        print(f'{label:<12} ERROR: {r.json().get("detail", "?")}')
        continue
    d = r.json()
    top = d['retrieval_top_score'] or 0
    src = d.get('refusal_source') or '-'
    print(f'{label:<12} {str(d["refused"]):<9} {src:<14} {top:<11.3f} {q[:40]}')

---
## 5 · Run the Groundedness Eval - `POST /eval`

Runs all rows in `data/golden_dataset.json` through `/answer` and computes five metrics.
The golden dataset was generated from real Qdrant chunks by `scripts/build_golden_dataset.py`.

Key concept: the eval harness scores each question on whether the system answered or refused
correctly, whether the answer was grounded, and whether citations were present.

Response shape:
```json
{
  "groundedness": 0.8,
  "citation_precision": 1.0,
  "citation_recall": 0.8,
  "false_answer_rate": 0.0,
  "false_refusal_rate": 0.2,
  "rows_scored": 10,
  "rows": [{ ... per-row detail ... }]
}
```

> This call hits the OpenAI API for every non-refused row. On the 10-row demo dataset it takes ~20-40 seconds.

In [ ]:
!curl -s -X POST http://localhost:8000/eval

In [ ]:
# POST /eval -- full groundedness eval with per-row breakdown
r = requests.post(f'{BASE}/eval')
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print(f'Rows scored        : {data["rows_scored"]}')
    print(f'Groundedness       : {data["groundedness"]*100:.1f}%')
    print(f'Citation precision : {data["citation_precision"]*100:.1f}%')
    print(f'Citation recall    : {data["citation_recall"]*100:.1f}%')
    print(f'False answer rate  : {data["false_answer_rate"]*100:.1f}%')
    print(f'False refusal rate : {data["false_refusal_rate"]*100:.1f}%')
    print()
    print('-- Row by row --')
    for row in data.get('rows', []):
        status = 'PASS' if row['passed'] else 'FAIL'
        src = f" [{row['refusal_source']}]" if row.get('refusal_source') else ''
        cites = ', '.join(row['cited_ids']) if row['cited_ids'] else 'none'
        print(f"  {status}  exp={row['expected']:<6} got={row['actual']:<8}{src}  cites=[{cites}]")
        print(f"       Q: {row['question'][:70]}")
        print(f"       A: {row['system_answer'][:70]}")
        print()

---
## 6 · Failure Mode - Question Too Short (422)

Pydantic validates `question` with `min_length=3`. A question shorter than 3 characters
returns **422** before any retrieval or API call is made -- no tokens spent.

This failure pattern is identical across all weeks: schema validation fires first.

In [ ]:
!curl -s -X POST http://localhost:8000/answer -H "Content-Type: application/json" -d "{\"question\": \"Hi\"}"

In [ ]:
# Failure: question too short -- Python
r = requests.post(f'{BASE}/answer', json={'question': 'Hi'})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects min_length=3, no API call made)')
print(json.dumps(r.json(), indent=2))

---
## 7 · Failure Mode - Missing Question Field (422)

Sending a body without the required `question` field also returns **422**.
FastAPI's Pydantic layer rejects missing required fields before the route handler runs.

In [ ]:
!curl -s -X POST http://localhost:8000/answer -H "Content-Type: application/json" -d "{}"

In [ ]:
# Failure: missing question field -- Python
r = requests.post(f'{BASE}/answer', json={})
print(f'Status: {r.status_code}  (expected 422 -- missing required field)')
detail = r.json().get('detail', [])
if isinstance(detail, list):
    for err in detail:
        print(f'  field: {err.get("loc")}, msg: {err.get("msg")}')
else:
    print(detail)

---
## 8 · Failure Mode - Golden Dataset Missing (500)

If `data/golden_dataset.json` is absent, the `/eval` endpoint returns **500** with a clear
message pointing to the fix command. The dataset is gitignored -- students who clone the
repo must run `python scripts/build_golden_dataset.py` before using `/eval`.

> To trigger this: rename `data/golden_dataset.json` temporarily, call `/eval`, then rename it back.

In [ ]:
# Show what the 500 body looks like when golden_dataset.json is missing.
# To really trigger: rename data/golden_dataset.json -> data/golden_dataset.json.bak, then:
#   r = requests.post(f'{BASE}/eval')
#   print(r.status_code, r.json())
print('Expected 500 body when data/golden_dataset.json is missing:')
print(json.dumps({
    'detail': 'Golden dataset missing: ./data/golden_dataset.json. Run: python scripts/build_golden_dataset.py'
}, indent=2))

---
## 9 · Full Raw Response Dump

Shows the complete JSON payload from `/answer` -- useful for debugging all diagnostic fields.

In [ ]:
# Full raw response -- shows retrieved_chunks, refusal_source, validation_detail etc.
r = requests.post(f'{BASE}/answer', json={'question': DEMO['bleu']})
print(json.dumps(r.json(), indent=2))

---
## 10 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs -- try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))